In [283]:
import pandas as pd
import yaml
import importlib
import etl.normalization as normalization

importlib.reload(normalization)


<module 'etl.normalization' from 'c:\\Users\\dergun\\Documents\\hfqa_tool\\etl\\normalization.py'>

In [ ]:
with open(r"C:\Users\dergun\Documents\hfqa_tool\hf_schema.yaml", "r", encoding="utf-8") as f:
    schema = yaml.safe_load(f)

schema

In [ ]:
df_raw = pd.read_excel(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008.xlsx",
    sheet_name=1,
    header=0
)
print(df_raw.shape)
df_raw.head(10)


In [ ]:
# Tranfrom Excel to Parquet (all to string)
#divide into meta and data
df_raw["row_type"] = ["meta"] * 7 + ["data"] * (len(df_raw) - 7)
df_raw_str = df_raw.astype("string[pyarrow]")
#convert to parquet
df_raw_str.to_parquet(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008_raw.parquet",
    index=False)



In [ ]:
# Load Parquet in as String 
df_raw_parquet_string = pd.read_parquet(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008_raw.parquet",
    dtype_backend="pyarrow"
).astype("string[pyarrow]")  
df_raw_parquet_string 


In [ ]:
import copy

# filter only columns starting with 'P'
parent_cols_spec = {
    name: spec
    for name, spec in schema["columns"].items()
    if name.startswith("P")
}

# make a full independent copy
schema_parent = copy.deepcopy(schema)
schema_parent["columns"] = parent_cols_spec

In [ ]:
print(schema_parent.keys())             # should include "columns"
print(schema_parent["columns"].keys())  

In [ ]:
normalize_dataframe = normaliyation.normalize_only_data_rows(df_raw_parquet_string,schema)


In [ ]:
normalize_dataframe